# 05 — National Analysis

**Workflow**
1. **Clip** all datasets to a country folder (`data/countries/{ISO3}/`) — run once
2. **Inspect** clipped data (maps, coverage)
3. **Single-epoch analysis** — concentration curve, CI, quintile breakdown
4. **Year-by-year time series** — CI and mean loss over time
5. **Epoch comparison** — overlay concentration curves across decades

To use a different population dataset, drop a GeoTIFF into `data/countries/{ISO3}/`
and set `POPULATION_FILE` in the configuration cell.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import geopandas as gpd
import rasterio
import rasterio.mask
from rasterio.warp import reproject, Resampling
from shapely.geometry import mapping
import yaml
from tqdm.notebook import tqdm

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from scripts.inequality import (
    calculate_CI, calculate_concentration_curve,
    calculate_quantile_ratio, prepare_arrays
)
from scripts.raster_utils import get_country_geometry

with open(ROOT / 'config' / 'config.yaml') as f:
    config = yaml.safe_load(f)

DATA = ROOT / 'data'
print('Ready.')

---
## Configuration
Edit this cell before running anything else.

In [ ]:
# --- Country ---
ISO3 = 'KEN'

# --- WBGT dataset ---
# Options: 'wbgt' | 'wbgt_baseline' | 'wbgt_future_2030_ssp245' | 'wbgt_future_2030_ssp585'
#          'wbgt_future_2050_ssp245' | 'wbgt_future_2050_ssp585'
WBGT_DATASET = 'wbgt'

# --- Population file ---
# Filename inside data/countries/{ISO3}/
# Default 'population.tif' is created by the clip step below.
# Add your own file to the folder and point to it here.
POPULATION_FILE = 'population.tif'

# --- Paths (used for clipping only — leave as-is unless data moved) ---
GLOBAL_POP_PATH = DATA / config['data']['population']
GLOBAL_RWI_PATH = DATA / config['data']['rwi']
BOUNDARIES_PATH = DATA / config['data']['boundaries']
ANNUAL_SRC_DIR  = DATA / 'processed' / 'annual' / WBGT_DATASET

# --- Country folder ---
COUNTRY_DIR      = DATA / 'countries' / ISO3
WBGT_COUNTRY_DIR = COUNTRY_DIR / WBGT_DATASET

print(f'Country:      {ISO3}')
print(f'WBGT dataset: {WBGT_DATASET}')
print(f'Population:   {POPULATION_FILE}')
print(f'Country dir:  {COUNTRY_DIR}')

---
## Step 1 — Clip datasets to country folder

Run once per country / WBGT dataset. Skips files that already exist unless `FORCE_CLIP = True`.

In [ ]:
FORCE_CLIP = False

COUNTRY_DIR.mkdir(parents=True, exist_ok=True)
WBGT_COUNTRY_DIR.mkdir(parents=True, exist_ok=True)

geom = get_country_geometry(str(BOUNDARIES_PATH), ISO3)


def clip_and_save(src_path: Path, dst_path: Path, geometry, force: bool = False) -> None:
    """Clip a raster to a geometry (reprojects geometry to raster CRS) and save."""
    if dst_path.exists() and not force:
        return
    with rasterio.open(src_path) as src:
        raster_crs = src.crs
        geo_s = gpd.GeoSeries([geometry], crs='EPSG:4326')
        clip_geom = [mapping(geo_s.to_crs(raster_crs).iloc[0])]
        out_image, out_transform = rasterio.mask.mask(src, clip_geom, crop=True)
        out_meta = src.meta.copy()
        out_meta.update({
            'height': out_image.shape[1],
            'width':  out_image.shape[2],
            'transform': out_transform,
            'compress': 'lzw',
        })
    with rasterio.open(dst_path, 'w', **out_meta) as dst:
        dst.write(out_image)


# Population
pop_dst = COUNTRY_DIR / 'population.tif'
clip_and_save(GLOBAL_POP_PATH, pop_dst, geom, force=FORCE_CLIP)
status = 'clipped' if not pop_dst.exists() or FORCE_CLIP else 'already exists'
print(f'Population:  {status}')

# RWI
rwi_dst = COUNTRY_DIR / 'rwi.tif'
clip_and_save(GLOBAL_RWI_PATH, rwi_dst, geom, force=FORCE_CLIP)
status = 'clipped' if not rwi_dst.exists() or FORCE_CLIP else 'already exists'
print(f'RWI:         {status}')

# Annual WBGT productivity rasters
annual_src_files = sorted(ANNUAL_SRC_DIR.glob('productivity_loss_*.tif'))
if not annual_src_files:
    print(f'WARNING: No annual rasters found in {ANNUAL_SRC_DIR}')
else:
    to_clip = [f for f in annual_src_files
               if not (WBGT_COUNTRY_DIR / f.name).exists() or FORCE_CLIP]
    print(f'WBGT:        clipping {len(to_clip)} / {len(annual_src_files)} rasters...')
    for f in tqdm(to_clip, desc='Clipping WBGT'):
        clip_and_save(f, WBGT_COUNTRY_DIR / f.name, geom, force=FORCE_CLIP)

print('\nCountry folder contents:')
for f in sorted(COUNTRY_DIR.rglob('*.tif')):
    size_mb = f.stat().st_size / 1e6
    print(f'  {str(f.relative_to(COUNTRY_DIR)):<55} {size_mb:.1f} MB')

---
## Step 2 — Load and align country data

In [ ]:
# Show available population files so you can pick one
print('Population files in country folder:')
for f in sorted(COUNTRY_DIR.glob('*.tif')):
    if 'population' in f.name or 'pop' in f.name.lower():
        print(f'  {f.name}')

wbgt_files = sorted(WBGT_COUNTRY_DIR.glob('productivity_loss_*.tif'))
years      = sorted([int(f.stem.split('_')[-1]) for f in wbgt_files])
print(f'\nAnnual WBGT rasters: {len(wbgt_files)}  ({min(years)}–{max(years)})')

In [ ]:
def read_raster(path: Path) -> tuple[np.ndarray, dict]:
    with rasterio.open(path) as src:
        arr    = src.read(1).astype(np.float32)
        nodata = src.nodata
        meta   = src.meta.copy()
    if nodata is not None:
        arr[arr == nodata] = np.nan
    return arr, meta


def resample_to(src_arr, src_meta, ref_meta, method=Resampling.bilinear):
    dest = np.full((ref_meta['height'], ref_meta['width']), np.nan, dtype=np.float32)
    reproject(source=src_arr, destination=dest,
              src_transform=src_meta['transform'], src_crs=src_meta['crs'],
              dst_transform=ref_meta['transform'], dst_crs=ref_meta['crs'],
              resampling=method, src_nodata=np.nan, dst_nodata=np.nan)
    return dest


def epoch_mean(wbgt_dir: Path, start: int, end: int) -> tuple[np.ndarray, dict] | None:
    """Average clipped annual rasters for years [start, end]."""
    files = [wbgt_dir / f'productivity_loss_{y}.tif'
             for y in range(start, end + 1)
             if (wbgt_dir / f'productivity_loss_{y}.tif').exists()]
    if not files:
        return None, None
    with rasterio.open(files[0]) as src:
        shape = (src.height, src.width)
        meta  = src.meta.copy()
    acc   = np.zeros(shape, np.float64)
    count = np.zeros(shape, np.int32)
    for f in files:
        with rasterio.open(f) as src:
            arr = src.read(1).astype(np.float64)
        valid = ~np.isnan(arr)
        acc[valid]   += arr[valid]
        count[valid] += 1
    return np.where(count > 0, acc / count, np.nan).astype(np.float32), meta


# Load static layers
pop, pop_meta = read_raster(COUNTRY_DIR / POPULATION_FILE)
rwi, rwi_meta = read_raster(COUNTRY_DIR / 'rwi.tif')
rwi[rwi < -10] = np.nan   # RWI nodata

# Resample RWI to pop grid if needed
if rwi_meta['transform'] != pop_meta['transform'] or rwi_meta['crs'] != pop_meta['crs']:
    rwi = resample_to(rwi, rwi_meta, pop_meta, Resampling.nearest)

print(f'Population grid: {pop.shape}  ({POPULATION_FILE})')
print(f'Total population: {np.nansum(pop[pop > 0]):,.0f}')

In [ ]:
# --- Set epoch for single-epoch analysis (Steps 3–4) ---
EPOCH_START = years[0]
EPOCH_END   = years[-1]
# -------------------------------------------------------

risk_arr, risk_meta = epoch_mean(WBGT_COUNTRY_DIR, EPOCH_START, EPOCH_END)
risk = resample_to(risk_arr, risk_meta, pop_meta)

arrays = prepare_arrays(pop, rwi, risk)
if arrays is None:
    raise ValueError('Insufficient valid data — check that pop, RWI and risk overlap.')
pop_f, rwi_f, risk_f = arrays

ci        = calculate_CI(pop_f, rwi_f, risk_f)
qr        = calculate_quantile_ratio(pop_f, rwi_f, risk_f)
mean_loss = float(np.average(risk_f, weights=pop_f))

print(f'Epoch {EPOCH_START}–{EPOCH_END}  |  {WBGT_DATASET}')
print(f'  CI:           {ci:.3f}')
print(f'  QR (Q5/Q1):   {qr:.2f}')
print(f'  Mean loss:    {mean_loss:.4f}')
print(f'  Population:   {pop_f.sum():,.0f}')

---
## Step 3 — Spatial maps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

pop_d  = np.where(pop  > 0,               pop,  np.nan)
risk_d = np.where(pop  > 0,               risk, np.nan)
rwi_d  = np.where((pop > 0) & ~np.isnan(rwi), rwi,  np.nan)

for ax, arr, cmap, vmin, vmax, title in [
    (axes[0], np.log1p(pop_d), 'viridis', None, None, f'Population (log)\n[{POPULATION_FILE}]'),
    (axes[1], risk_d,          'YlOrRd',  0,    None, f'Mean productivity loss\n{EPOCH_START}–{EPOCH_END}'),
    (axes[2], rwi_d,           'RdYlGn', -2,    2,    'Relative Wealth Index'),
]:
    im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

fig.suptitle(f'{ISO3} — {WBGT_DATASET}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 4 — Concentration curve and quintile breakdown

In [ ]:
cum_pop, cum_risk = calculate_concentration_curve(pop_f, rwi_f, risk_f)

# Quintile breakdown
df = pd.DataFrame({'pop': pop_f, 'rwi': rwi_f, 'risk': risk_f}).sort_values('rwi')
df['cum_pop'] = df['pop'].cumsum()
tp = df['pop'].sum()
qdf = pd.DataFrame([
    {'Q': f'Q{q}', 'loss': np.average(
        df[(df['cum_pop'] > (q-1)/5*tp) & (df['cum_pop'] <= q/5*tp)]['risk'],
        weights=df[(df['cum_pop'] > (q-1)/5*tp) & (df['cum_pop'] <= q/5*tp)]['pop']
    )} for q in range(1, 6)
])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.plot(cum_pop, cum_risk, color='steelblue', linewidth=2,
        label=f'CI = {ci:.3f}  |  QR = {qr:.2f}')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Line of equality')
ax.fill_between(cum_pop, cum_risk, cum_pop,
                alpha=0.15, color='steelblue' if ci < 0 else 'tomato')
ax.set_xlabel('Cumulative population share (poorest → wealthiest)')
ax.set_ylabel('Cumulative productivity loss share')
ax.set_title(f'Concentration Curve — {ISO3} ({EPOCH_START}–{EPOCH_END})')
ax.legend()
ax.set_xlim(0, 1); ax.set_ylim(0, 1)

ax = axes[1]
ax.bar(qdf['Q'], qdf['loss'],
       color=plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(qdf))),
       edgecolor='white')
ax.axhline(mean_loss, color='black', linewidth=1.2, linestyle='--',
           label=f'Mean ({mean_loss:.4f})')
ax.set_xlabel('Wealth quintile (Q1 = poorest)')
ax.set_ylabel('Mean annual productivity loss')
ax.set_title('Loss by Wealth Quintile')
ax.legend()

plt.tight_layout()
plt.show()
print(qdf.to_string(index=False))

---
## Step 5 — Year-by-year time series

In [ ]:
ts_records = []

for fpath in tqdm(wbgt_files, desc='Annual CI'):
    year = int(fpath.stem.split('_')[-1])
    risk_yr, risk_yr_meta = read_raster(fpath)
    risk_yr_res = resample_to(risk_yr, risk_yr_meta, pop_meta)
    arr = prepare_arrays(pop, rwi, risk_yr_res)
    if arr is None:
        continue
    p, w, r = arr
    ts_records.append({
        'year':      year,
        'CI':        calculate_CI(p, w, r),
        'QR':        calculate_quantile_ratio(p, w, r),
        'mean_loss': float(np.average(r, weights=p)),
    })

ts_df = pd.DataFrame(ts_records).sort_values('year')

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(ts_df['year'], ts_df['mean_loss'], marker='o', markersize=3, color='tomato')
axes[0].set_ylabel('Mean annual productivity loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(ts_df['year'], ts_df['CI'], marker='o', markersize=3, color='steelblue')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_ylabel('Concentration Index')
axes[1].set_xlabel('Year')
axes[1].grid(True, alpha=0.3)

fig.suptitle(f'{ISO3} — {WBGT_DATASET}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

ts_df

---
## Step 6 — Epoch comparison

Define as many epochs as you like. Each gets its own concentration curve.

In [ ]:
# --- Define epochs to compare ---
EPOCHS = {
    'Early  (1983–1999)': (1983, 1999),
    'Late   (2000–2016)': (2000, 2016),
    'Full   (1983–2016)': (1983, 2016),
}
# --------------------------------

colors = cm.tab10(np.linspace(0, 0.5, len(EPOCHS)))
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
epoch_summary = []

for (label, (start, end)), color in zip(EPOCHS.items(), colors):
    r_arr, r_meta = epoch_mean(WBGT_COUNTRY_DIR, start, end)
    if r_arr is None:
        print(f'  {label}: no data')
        continue
    r_res = resample_to(r_arr, r_meta, pop_meta)
    arr   = prepare_arrays(pop, rwi, r_res)
    if arr is None:
        continue
    p, w, r = arr
    ep_ci   = calculate_CI(p, w, r)
    ep_loss = float(np.average(r, weights=p))
    cp, cr  = calculate_concentration_curve(p, w, r)
    epoch_summary.append({'Epoch': label.strip(), 'CI': ep_ci, 'Mean loss': ep_loss})

    axes[0].plot(cp, cr, color=color, linewidth=2,
                 label=f'{label.strip()}  CI={ep_ci:.3f}')

axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Equality')
axes[0].set_xlabel('Cumulative population share (poorest → wealthiest)')
axes[0].set_ylabel('Cumulative productivity loss share')
axes[0].set_title(f'{ISO3} — Concentration curves by epoch')
axes[0].legend(fontsize=9)
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1)

ep_df = pd.DataFrame(epoch_summary)
x = np.arange(len(ep_df))
axes[1].bar(x, ep_df['CI'], color=colors[:len(ep_df)], edgecolor='white')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_xticks(x)
axes[1].set_xticklabels(ep_df['Epoch'], rotation=15, ha='right')
axes[1].set_ylabel('Concentration Index')
axes[1].set_title('CI by epoch')

plt.tight_layout()
plt.show()

print(ep_df.to_string(index=False))